# 📱 Mobile Price Range Classification

Predicting a mobile phone's **price range** (Low / Medium / High / Very High) from its technical specifications, using both a classical ML model (Random Forest) and a Deep Learning model (Keras Neural Network).

**Dataset:** 2,000 phones, 20 spec features + `price_range` target (4 classes), no missing values.


## 1. Imports

In [26]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , StandardScaler
from sklearn.metrics import accuracy_score , classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Dropout

## 2. Load & Explore Data

In [3]:
df = pd.read_csv("mobile (2).csv")
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [7]:
df.isna().sum()

battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
price_range      0
dtype: int64

## 3. Preprocessing

Encode any categorical columns, split into train/test, and scale features (fit only on train to avoid data leakage).

In [8]:
encode = LabelEncoder()
for i in df.select_dtypes("str"):
    df[i] = encode.fit_transform(df[i])

In [9]:
x = df.drop("price_range",axis=1)
y = df["price_range"]

In [13]:
x_train , x_test , y_train , y_test = train_test_split(x,y,random_state=42,test_size=0.2)

In [14]:
scale = StandardScaler()
x_trian_scaled = scale.fit_transform(x_train)
x_test_scaled = scale.transform(x_test)

## 4. Model 1 — Random Forest Classifier

A fast, interpretable baseline model.

In [15]:
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(x_trian_scaled,y_train)
y_pred_rf = model_rf.predict(x_test_scaled)

In [17]:
print(accuracy_score(y_test,y_pred_rf))
print(classification_report(y_test,y_pred_rf))

0.8925
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       105
           1       0.89      0.87      0.88        91
           2       0.78      0.87      0.82        92
           3       0.94      0.87      0.90       112

    accuracy                           0.89       400
   macro avg       0.89      0.89      0.89       400
weighted avg       0.90      0.89      0.89       400



## 5. Model 2 — Deep Neural Network (Keras)

A `Dense(200) → Dropout → Dense(150) → Dropout → Dense(100) → Dropout → Dense(4, softmax)` network. Dropout layers help prevent overfitting, and `sparse_categorical_crossentropy` is used since the target is multi-class and integer-encoded.

In [27]:
model = Sequential()
model.add(Dense(200,activation='relu',input_dim=20))
model.add(Dropout(0.3))
model.add(Dense(150,activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(100,activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(4,activation='softmax'))

model.compile(
    optimizer = "adam",
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']

)

model.fit(x_trian_scaled,y_train,epochs=12,batch_size=32,validation_split=0.2)

Epoch 1/12


c:\Users\snid\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3781 - loss: 1.2932 - val_accuracy: 0.6500 - val_loss: 1.0441
Epoch 2/12
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6055 - loss: 0.8916 - val_accuracy: 0.7375 - val_loss: 0.6169
Epoch 3/12
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7281 - loss: 0.6107 - val_accuracy: 0.8625 - val_loss: 0.4064
Epoch 4/12
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7852 - loss: 0.4797 - val_accuracy: 0.8438 - val_loss: 0.3771
Epoch 5/12
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8133 - loss: 0.4399 - val_accuracy: 0.8719 - val_loss: 0.3007
Epoch 6/12
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8336 - loss: 0.3752 - val_accuracy: 0.8781 - val_loss: 0.2791
Epoch 7/12
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8445 - loss: 0.3572 - val_accuracy: 0.9031 - val_loss: 0.2352
Epoch 8/12
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8539 - loss: 0.3396 - val_accuracy: 0.9000 - val_loss: 0.2332
Epo

## 6. Evaluate Neural Network on Test Data

In [28]:
test_loss , test_accuracy = model.evaluate(x_test_scaled,y_test)
print("Test Loss",test_loss)
print("Test Accuracy",test_accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9000 - loss: 0.2297 
Test Loss 0.22967155277729034
Test Accuracy 0.8999999761581421


## 💼 Business Use Case

A model like this has direct value in the mobile phone industry:

- **Product pricing strategy**: Before launch, a manufacturer can feed a prototype's specs into the model to see which price segment it naturally falls into, helping validate or adjust pricing decisions before going to market.
- **Competitive benchmarking**: Retailers and e-commerce platforms can auto-tag new phone listings into price tiers, even when the listed price is missing, inconsistent, or needs sanity-checking against the phone's actual specs.
- **Spec-to-price optimization**: R&D teams can use the model to answer "what-if" questions, e.g. how much does increasing RAM or battery capacity push a phone into a higher price bracket, to optimize the specs-to-price ratio for a target market segment.
- **Customer-facing tools**: A simplified version could power a "what's a fair price for this phone?" feature in a shopping app, increasing customer trust.

In short, this isn't just an academic classification exercise, it's a decision-support tool for pricing and product strategy.


## ✅ Conclusion

| Model | Test Accuracy |
|---|---|
| Random Forest | 89.25% |
| Neural Network | 90.00% |

Both models perform comparably well:

- The **Random Forest** model is a strong, fast, easily interpretable baseline that requires no feature scaling and gives clear per-class insight via `classification_report`.
- The **Neural Network** slightly outperforms it and, thanks to dropout regularization, generalizes reliably to unseen data (train, validation, and test accuracy all landed within ~1-2% of each other, indicating no significant overfitting).

For a production setting, the Random Forest model is a reasonable first choice given its simplicity and interpretability, while the neural network leaves more headroom for improvement as more data becomes available.

**Possible next steps:**
- Hyperparameter tuning (GridSearch/RandomSearch for Random Forest; architecture/learning-rate search for the NN).
- Add `EarlyStopping` and `ModelCheckpoint` callbacks to the NN training loop.
- Feature importance analysis to identify which specs most influence price range.
- Deploy the best model behind a simple API (e.g., Flask/FastAPI) for real-time predictions.
